In [9]:
%pip install matplotlib statsmodels scipy pandas numpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

# 1. Cargar la base de datos
df = pd.read_csv("abandono_producto_financiero.csv")

# 2. Configurar los factores categóricos con sus niveles y referencias fijas
# Referencia de pais: Francia | Niveles: Francia, Alemania, España
df["pais"] = pd.Categorical(df["pais"], categories=["Francia", "Alemania", "España"])

# Referencia de sexo: Mujer | Niveles: Mujer, Hombre
df["sexo"] = pd.Categorical(df["sexo"], categories=["Mujer", "Hombre"])

# 3. Inspeccionar las primeras filas y resumen
print("Forma de la base de datos (filas, columnas):", df.shape)
print("\nTipos de datos de las variables:")
print(df.dtypes)
print("\nPrimeras 5 filas:")
df.head()


Note: you may need to restart the kernel to use updated packages.
Forma de la base de datos (filas, columnas): (10000, 14)

Tipos de datos de las variables:
numero_fila              int64
id_cliente               int64
apellido                   str
puntaje_crediticio       int64
pais                  category
sexo                  category
edad                     int64
antiguedad               int64
saldo                  float64
numero_productos         int64
tiene_tarjeta            int64
miembro_activo           int64
salario_estimado       float64
abandono                 int64
dtype: object

Primeras 5 filas:


,numero_fila,id_cliente,apellido,puntaje_crediticio,pais,sexo,edad,antiguedad,saldo,numero_productos,tiene_tarjeta,miembro_activo,salario_estimado,abandono
0,1,15634602,Hargrave,619,Francia,Mujer,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,España,Mujer,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,Francia,Mujer,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,Francia,Mujer,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,España,Mujer,43,2,125510.82,1,1,1,79084.10,0


Resumen Exploratorio de la Base de Datos: Abandono de Productos Financieros

El conjunto de datos contiene información socio-demográfica y financiera de los clientes de una entidad bancaria, orientada a analizar la fuga o abandono (churn) de productos financieros.

1. Observaciones Iniciales (Muestra de Primeras 5 Filas)

Diversidad Geográfica y Demográfica: La muestra inicial evidencia clientes residentes en países como Francia y España, con edades maduras (rango entre 39 y 43 años en los registros observados).

Saldos nulos: Se observa presencia de registros con saldo igual a 0.00 (p. ej., clientes con ID 15634602 y 15701354), lo que sugiere la necesidad de tratar o analizar por separado a los clientes con cuentas inactivas o con saldo en cero.

Variable Objetivo (abandono): En las primeras 5 observaciones se registran 2 casos de abandono (abandono = 1) y 3 permanencias (abandono = 0), lo cual servirá de base para calibrar los modelos predictivos posteriores.

In [10]:
# Frecuencias y proporciones de la variable respuesta (abandono)
tabla_abandono = pd.crosstab(df["abandono"], columns="conteo")
tabla_abandono["proporcion"] = df["abandono"].value_counts(normalize=True)
print("Distribución de la variable respuesta (abandono):")
print(tabla_abandono)

# Resumen estadístico de variables continuas
print("\nResumen descriptivo de variables cuantitativas:")
print(df[["puntaje_crediticio", "edad", "antiguedad", "saldo", "numero_productos", "salario_estimado"]].describe())

Distribución de la variable respuesta (abandono):
col_0     conteo  proporcion
abandono                    
0           7963      0.7963
1           2037      0.2037

Resumen descriptivo de variables cuantitativas:
       puntaje_crediticio          edad    antiguedad          saldo  \
count        10000.000000  10000.000000  10000.000000   10000.000000   
mean           650.528800     38.921800      5.012800   76485.889288   
std             96.653299     10.487806      2.892174   62397.405202   
min            350.000000     18.000000      0.000000       0.000000   
25%            584.000000     32.000000      3.000000       0.000000   
50%            652.000000     37.000000      5.000000   97198.540000   
75%            718.000000     44.000000      7.000000  127644.240000   
max            850.000000     92.000000     10.000000  250898.090000   

       numero_productos  salario_estimado  
count      10000.000000      10000.000000  
mean           1.530200     100090.239881  
std 

## Sección 2: Análisis Univariado

### 1. Variable Respuesta (`abandono`)
* **Distribución:** La cartera muestra que el **79.63%** (7,963 clientes) permanece en el banco (`abandono = 0`), mientras que el **20.37%** (2,037 clientes) cerró su relación financiera (`abandono = 1`).
* **Implicación Metodológica:** Existe un desbalance moderado de clases (~80/20). Evaluar únicamente la exactitud (*accuracy*) resultaría engañoso, ya que un modelo ingenuo que prediga que nadie se va tendría ~80% de aciertos pero 0% de detección de fuga. Por ello, será necesario evaluar la **sensibilidad** y calibrar el umbral de probabilidad (p. ej. 0.4 o 0.5) para gestionar eficazmente la retención.

### 2. Variables Cuantitativas

* **Edad (`edad`):** La edad media de la cartera es de **38.9 años**, con un 50% central de los clientes ubicados entre los **32 y 44 años** (IQR). El rango abarca desde los 18 hasta los 92 años.
* **Saldo en Cuenta (`saldo`):** El promedio se ubica en **$76,485.89**. Destaca que al menos un **25% de los clientes registra un saldo de $0.00**, mientras que el 50% con mayor saldo alcanza hasta los **$250,898.09**.
* **Puntaje Crediticio (`puntaje_crediticio`):** Presenta una media de **650.5 puntos** (rango entre 350 y 850), reflejando una distribución centrada y estable en la calidad crediticia.
* **Antigüedad (`antiguedad`):** La relación promedio con la entidad es de **5.01 años**, distribuida de forma homogénea en el rango de 0 a 10 años.
* **Número de Productos (`numero_productos`):** Concentración alta en pocos productos; la mediana es **1 producto** y el 75% de la muestra mantiene máximo 2 productos (rango de 1 a 4).
* **Salario Estimado (`salario_estimado`):** Presenta un comportamiento cercano a una distribución uniforme entre **$11.58 y $199,992.48**, con un promedio de **$100,090.24**.